# Harness Engineering: OpenCode Case Study
## Chapter 1: Overall Harness

In earlier labs, you built an agent loop and connected it to tools. Now we will see how these ideas work together in **OpenCode**, an open-source coding agent that can read files, edit code, and run commands.

We will follow one small repair. A program averages three grades: 80, 100, and a missing score. The requirement says to ignore missing scores, so the answer should be 90. The program returns 60. As OpenCode works, we will examine what the model requests, how OpenCode carries it out, and how we check the result.

Follow five questions:
1. Where is the bug?
2. What does OpenCode actually do?
3. How does a tool result help it choose the next step?
4. Did it really fix the program?
5. How can a later conversation continue the same task?

**Part 1** follows the repair. **Part 2** explains the harness around the model. **Part 3** traces the key steps in real source code.


### Part 1: Follow the Repair
#### 1. Where is the bug?

Start with the project, before asking a model to change it. Run the setup cell in a fresh Colab runtime. It installs OpenCode `1.18.34` and makes a new copy of the example; earlier copies are left untouched. No model is called yet.

The directory below shows what each file is for. Read the requirements, program, data, and test before running them.


In [ ]:
#@title Prepare OpenCode and the example project
from pathlib import Path
import subprocess
import sys
import tempfile

REPO_URL = "https://github.com/SleepyLGod/AIST4020-Lab-Tutorials.git"
LAB_RELATIVE = Path("labs/opencode/overall-harness")
LAB_DIR = next((p for p in (Path.cwd(), Path.cwd() / LAB_RELATIVE)
                if (p / "lab_support.py").is_file()), None)
if LAB_DIR is None:
    course = Path(tempfile.mkdtemp(prefix="aist4020-course-"))
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(course)], check=True)
    LAB_DIR = course / LAB_RELATIVE
sys.path.insert(0, str(LAB_DIR))
import lab_support as support
if Path(support.__file__).resolve() != (LAB_DIR / "lab_support.py").resolve():
    raise RuntimeError("Restart the runtime to load this lab's helper.")
lab = support.open_lab(LAB_DIR, version="1.18.34")


In [ ]:
support.show_project(lab.project)
for name in ("README.md", "grades.py", "scores.csv", "test_grades.py"):
    support.show_details(name, (lab.project / name).read_text(), opened=True)


Run the program and its test yourself. The requirement says 90; compare that with the program's actual output. The test should fail before the repair.


In [ ]:
initial_average = lab.check([
    sys.executable, "-c",
    "from grades import average; print('Average:', average('scores.csv'))",
])
TEST_COMMAND = ["python3", "-m", "unittest", "-v"]
initial_test = lab.check(TEST_COMMAND)
assert not initial_test.timed_out and initial_test.returncode != 0
assert "60.0 != 90.0" in initial_test.stdout + initial_test.stderr
before = support.snapshot(lab.project)


In `grades.py`, `scores` contains two numbers, but `len(rows)` counts all three rows. That makes the calculation 180 / 3. The requirement calls for counting only the recorded scores.

#### 2. What does OpenCode actually do?

Now let OpenCode work on the same problem. Choose one model below and set `ENABLE_MODEL_CALLS = True`. The configuration step uses Colab Secrets or hidden input for API keys.

DeepSeek is the default; NVIDIA and a custom endpoint are alternatives. Hosted calls can incur charges, and one repair can make several model calls. Ollama runs inside Colab; use a GPU for that option.


In [ ]:
BACKENDS = {
    "deepseek": dict(base_url="https://api.deepseek.com/v1",
                     model="deepseek-flash", key_env="DEEPSEEK_API_KEY"),
    "nvidia": dict(base_url="https://integrate.api.nvidia.com/v1",
                   model="z-ai/glm-5.3", key_env="NVIDIA_NIM_API_KEY"),
    "ollama": dict(base_url="http://127.0.0.1:11434/v1",
                   model="qwen3:8b", key_env=""),
    "custom": dict(base_url="", model="", key_env=""),
}

backend = "deepseek"
ENABLE_MODEL_CALLS = False


Before the repair, allow OpenCode to read and search files, edit only `grades.py`, and run our test command. The `"*": "deny"` rule denies other tool actions.

These are tool permissions, not a separate protected computer. Use the supplied disposable project.


In [ ]:
PERMISSIONS = {
    "*": "deny",
    "read": "allow",
    "glob": "allow",
    "grep": "allow",
    "edit": {"*": "deny", "grades.py": "allow", "**/grades.py": "allow"},
    "bash": {"*": "deny", "python3 -m unittest -v": "allow"},
    "external_directory": "deny",
}
lab.configure(backend, BACKENDS, permissions=PERMISSIONS, enabled=ENABLE_MODEL_CALLS)


Start with a greeting to check the connection. This request cannot use tools. Resolve any connection error before moving on.

`lab.run(...)` is our course helper, not an OpenCode Python API. With the default model, it runs:

```bash
opencode run --pure --format json --agent build --model course/deepseek-flash \
  "Reply with one sentence: OpenCode is ready for the lab. Do not use tools."
```

It prints the actual command and working folder before each run.


In [ ]:
hello = lab.run(
    "Reply with one sentence: OpenCode is ready for the lab. Do not use tools.",
    tools=False, label="hello",
)


Read the task below. It asks OpenCode to test, fix, and test again. It also asks for the test result first in the final report; we will return to that instruction.

Run it and follow the actions. Which file does it read first? Does it run the test before changing the code?

In this lab, OpenCode asks the model to wrap up on round 12; the helper stops a run after five minutes.


In [ ]:
repair_prompt = """Read README.md and grades.py. Run python3 -m unittest -v to reproduce
the bug. Fix only grades.py, then rerun the same test command.
Report what changed and what the test actually returned.
For this session, put the test result before the explanation of the fix."""

repair = lab.run(repair_prompt)
print("Repair session:", repair.session_id or "not reported")


#### 3. How does a tool result help it choose the next step?

The table lists the actions reported in your run, in order. Find a test command, expand its output, and look at the next action. Did a failed test lead to an edit? Was there another test after that edit?

A tool marked `completed` has finished running; its test may still have failed. If your run stopped early, use its last action or error to see where it stopped.


In [ ]:
support.show_tool_actions(repair)


#### 4. Did it really fix the program?

Compare the code diff, then run the checks yourself. The table compares these checks with the ones you ran before the repair. Check the average, the test, and which files changed.

These are your checks, not the agent's. Use the action table above to see whether the agent also retested.


In [ ]:
changed_files = support.show_changes(before, support.snapshot(lab.project), lab.redact)

test_after = lab.check(TEST_COMMAND)

# Check the expected value without relying on the project's test file.
average_check = lab.check([
    sys.executable, "-c",
    "from grades import average; actual = average('scores.csv'); "
    "print('Average:', actual); assert actual == 90.0, actual",
])
support.show_repair_comparison(
    initial_average, initial_test, average_check, test_after, changed_files, lab.redact,
)


Read the final answer alongside the actions from Question 3. If it says the tests passed, find the test **after the last edit**. If only your own test passed, the program may be fixed even though the agent skipped that requested step.

You now have three things to compare: the changed code, the test results, and what OpenCode said it did.


#### 5. How can a later conversation continue the same task?

Now continue the conversation about the repair. Ask, "What report order did I request?" once in the repair session and once in a new session. A **session** is one saved conversation. The model and project stay the same.

Passing `session_id` adds `--session` to the command; leaving it out starts a new conversation. Here, `tools=False` selects a configuration that denies tool use, so neither request can look for the answer in files.


In [ ]:
if not repair.session_id:
    raise RuntimeError("The repair returned no session ID. Check its error before continuing.")

history_question = """What order did I ask you to use in your report earlier in this session?
If you cannot find an earlier request in this conversation, say so.
Do not read or change project files."""

same = lab.run(history_question, session_id=repair.session_id, tools=False, label="same")
fresh = lab.run(history_question, tools=False, label="fresh")
assert same.session_id == repair.session_id
assert fresh.session_id and fresh.session_id != same.session_id


Compare the two replies side by side below. Does the continued session recall your requested order? How does the new session answer?

The last row checks whether your original instruction appears in the saved user messages. Expand those messages to read it in context; a correct reply alone does not show where the answer came from.


In [ ]:
continued_history = lab.export(same.session_id, label="continued", show=False)
fresh_history = lab.export(fresh.session_id, label="fresh", show=False)

instruction = "For this session, put the test result before the explanation of the fix."
support.show_session_comparison(
    same, fresh, continued_history=continued_history, fresh_history=fresh_history,
    instruction=instruction, redact=lab.redact,
)
assert support.has_user_message(continued_history, instruction)
assert not support.has_user_message(fresh_history, instruction)


### Part 2: What Is the Harness?

The model did not receive only "fix this bug." It worked with a request, project files, tools, permission rules, and a conversation that grew as the task progressed.

> A harness is the setup around the model that helps an agent do a task more reliably: files, instructions, tools, action rules, session history, memory notes, checks, and feedback loops.

In this case, OpenCode supplies the tools and conversation handling. We supply the project, task, permissions, and independent tests. We used session history, not an additional memory system.


#### Connect the example to the harness

Follow a test result around the loop: OpenCode adds it to the conversation, then the model can request another action or answer. The diagram shows how the loop works; the action table in Part 1 shows what happened in your run.

![OpenCode prepares model input, checks tool requests, runs allowed actions, and returns their results to the next model call.](https://raw.githubusercontent.com/SleepyLGod/AIST4020-Lab-Tutorials/main/labs/opencode/overall-harness/assets/agent-loop.png)

| In this example | What it does | Name |
| --- | --- | --- |
| The four project files | Give the agent a program, data, requirements, and a test to work with. | Workspace |
| "Fix only grades.py, then rerun the same test command." | Tells the model what to do. | Instructions |
| `read` returns file contents; `bash` returns test output. | Makes information from the project available to later model calls. Look for these results in your action table. | Tools and context |
| `PERMISSIONS` allows editing `grades.py` and running the specified test command. | Checks which requested actions may run. | Action rules |
| A failed test reports `60.0 != 90.0`. | Gives the model a concrete problem to address if it runs the test. | Feedback |
| The saved conversation contains the report-order request and tool results. | Lets a later turn continue the same task. | Session history |
| You rerun the test and inspect the diff. | Checks the repair against the requirement and the allowed file changes. | Independent checks |

Seeing a file in the notebook is not the same as sending it to the model. A tool can read it and return its contents. Likewise, asking "only change grades.py" gives an instruction; the permission rules check actions in code.

A **session** is the saved conversation. **Context** is what the model receives for one call: instructions, the current task, tool descriptions, and selected earlier messages or results. For longer tasks, OpenCode may summarize older messages before sending them to the model.


#### How do we improve it?

The test output gives feedback during a task: after seeing a failure, the agent can change the code and test again. We can also use the records to improve the next run.

Suppose the program is fixed but the agent did not retest. Check what happened before changing anything: did it stop, reach a time limit, or try a command that was denied? Then change one relevant part of the setup and rerun from a fresh project.

That is harness engineering: improve the instructions, tools, rules, or checks around the model, and see whether the change helps. It does not require changing the model itself.

For further reading, Böckeler describes guidance before actions and feedback afterward in [Harness engineering for coding agent users](https://martinfowler.com/articles/harness-engineering.html). OpenAI gives a larger project example in [Harness engineering](https://openai.com/index/harness-engineering/).


### Part 3: Find the Same Steps in the Source

We will follow the request, a tool result, and the saved conversation. The project and independent tests are our course code; the excerpts below show the parts OpenCode performs.

Sources use [OpenCode v1.18.34](https://github.com/anomalyco/opencode/tree/v1.18.34), matching the lab. `// ...` omits unrelated lines. In this TypeScript code, read `yield*` as "perform this operation and get its result"; focus on the values passed between calls.


#### 1. Send the task with its tools

Your repair request enters OpenCode with the selected model and session:

```ts
const model = pick(args.model)
const result = await client.session.prompt({
  sessionID,
  agent,
  model,
// ...
  parts: [...files, { type: "text", text: message }],
})
```

*Source: [cli/cmd/run.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/cli/cmd/run.ts#L863)*

`message` is the task you wrote; `sessionID` identifies its conversation. Later, the model call receives the prepared messages and tool descriptions:

```ts
result: streamText({
// ...
  tools: prepared.tools,
// ...
  messages: prepared.messages,
  model: wrapLanguageModel({
    model: language,
// ...
  }),
// ...
}),
```

*Source: [session/llm.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/llm.ts#L280)*

`prepared.messages` contains the input for this call. `prepared.tools` describes the actions the model can request. After a tool runs, its result can be included in the messages for a later call.


#### 2. Turn a request into an action

For a read request, `args` contains a file path such as the path to `grades.py`. OpenCode calls the selected tool's implementation:

```ts
        const result = yield* item.execute(args, ctx)
```

*Source: [session/tools.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/tools.ts#L111)*

`ctx` gives the tool access to the current session and permission checks. Inside the read tool, the permission check comes before the file read:

```ts
yield* ctx.ask({
  permission: "read",
  patterns: [path.relative(instance.worktree, filepath)],
// ...
})
// ...
const file = yield* lines(filepath, { limit: params.limit ?? DEFAULT_READ_LIMIT, offset: params.offset || 1 })
// ...
return {
  title,
  output,
// ...
}
```

*Source: [tool/read.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/tool/read.ts#L255)*

`ctx.ask` checks whether reading the path is allowed. If allowed, `lines(...)` reads the file, and `output` holds the returned content.

The test command uses the `bash` tool instead. Both return results to the conversation. The next chapter examines the permission rules in more detail.


#### 3. Keep the test result with its request

Suppose the test reports `60.0 != 90.0`. OpenCode receives the tool result and completes its record:

```ts
case "tool-result": {
// ...
  yield* completeToolCall(value.id, output)
  return
}
```

*Source: [session/processor.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/processor.ts#L383)*

Inside `completeToolCall`, it finds the request and saves the returned output:

```ts
const match = yield* readToolCall(toolCallID)
if (!match || match.part.state.status !== "running") return
yield* session.updatePart({
  ...match.part,
  state: {
    status: "completed",
    input: match.part.state.input,
    output: output.output,
// ...
  },
})
```

*Source: [session/processor.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/processor.ts#L169)*

`toolCallID` keeps each result attached to the right request. `output.output` holds the returned text. The status becomes `completed` because the tool finished, even when the test failed.

OpenCode also writes these records to its database. You can follow that storage code in [session/session.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/session.ts#L637) and [session/projector.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/core/src/session/projector.ts#L310).


#### 4. Use the result in the next model call

At the start of each loop pass, OpenCode loads messages for the current session:

```ts
let msgs = yield* MessageV2.filterCompactedEffect(sessionID).pipe(
  Effect.provideService(Database.Service, database),
)
```

*Source: [session/prompt.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/prompt.ts#L1092)*

`msgs` is the loaded conversation. It can now include the test output saved above. The function also handles history that has been shortened.

This is also why continuing a session matters: the same `sessionID` lets OpenCode load your earlier request to put the test result first. A new session does not load that conversation.

Next, this call converts `msgs` into messages for the model. Its result is stored as `modelMsgs`:

```ts
MessageV2.toModelMessagesEffect(msgs, model),
```

*Source: [session/prompt.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/prompt.ts#L1262)*

Inside that function, a completed tool request contributes its result:

```ts
if (part.state.status === "completed") {
// ...
  assistantMessage.parts.push({
    type: ("tool-" + part.tool) as `tool-${string}`,
    state: "output-available",
    toolCallId: part.callID,
    input: part.state.input,
    output,
// ...
  })
}
```

*Source: [session/message-v2.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/message-v2.ts#L296)*

`toolCallId` identifies the request, and `output` carries its result. The prepared conversation, `modelMsgs`, is passed into the next model step:

```ts
const result = yield* handle.process({
// ...
  system,
  messages: [
    ...modelMsgs,
// ...
  ],
  tools,
  model,
// ...
})
```

*Source: [session/prompt.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/prompt.ts#L1272)*

The path is now complete: **test output → saved tool result → loaded conversation → next model input**. The model can use the failure to choose another action. Loading a result makes it available; it does not guarantee the model will make the right choice.

OpenCode's CLI reports finished or failed tool actions as JSON. The notebook reads those records to display each request and its result. [Source: CLI tool events](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/cli/cmd/run.ts#L724).


#### What to remember

The model proposes the next action. OpenCode connects that request to tools, checks the action, and returns the result through the conversation. The harness includes this loop and the files, instructions, rules, state, and checks around it.

Our small repair lets you follow each part. More complicated tasks use the same basic ideas, but need stronger guidance and more thorough checks.

#### References

- [OpenCode CLI](https://opencode.ai/docs/cli/), [rules](https://opencode.ai/docs/rules/), and [permissions](https://opencode.ai/docs/permissions/).
- [OpenCode v1.18.34 source](https://github.com/anomalyco/opencode/tree/v1.18.34).
- [Birgitta Böckeler: Harness engineering for coding agent users](https://martinfowler.com/articles/harness-engineering.html).
- [OpenAI: Harness engineering](https://openai.com/index/harness-engineering/).

#### Optional: Keep your run

The helper saves command output and exported conversations. To download them, run the cell below. Known keys are masked; review file contents and personal text before sharing.

When finished, use **Runtime → Disconnect and delete runtime** to release the Colab machine. Download first if you want to keep the records.


In [ ]:
DOWNLOAD_RECORDS = False
if DOWNLOAD_RECORDS:
    lab.save_record({
        "repair_session": repair.session_id,
        "fresh_session": fresh.session_id,
        "changed_files": changed_files,
        "your_test_passed": test_after.returncode == 0 and not test_after.timed_out,
        "average_is_90": average_check.returncode == 0 and not average_check.timed_out,
    })
    lab.download_records()
